# Certifying an exponential decay rate — the minimal pipeline

This notebook certifies a **guaranteed exponential decay rate** `alpha` for a
damped pendulum on a box `Q_R` around the origin, using `pyddrv.verify_stability`.

It exercises the **pure-NumPy** path (the pendulum's `np.sin` field does not
JIT-trace, so no JAX is needed — same certificate, just slower) and the
**extreme-value (EVT) Lipschitz estimator**, which is the sound choice here
because the pendulum's Jacobian is *not* affine in the state.

> Prerequisite: `pip install "pyddrv[jax,dev] @ git+https://github.com/NetDLab/pyddrv"`
> (or a dev checkout on your `PYTHONPATH`). See `GETTING_STARTED.md`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pyddrv import verify_stability
from pyddrv.systems import damped_pendulum

## 1. The system

The damped pendulum in state `x = (theta, omega)`:

$$\dot\theta = \omega, \qquad \dot\omega = -\sin\theta - \omega.$$

`pyddrv` fields are **batched**: they map a stack of states `(N, d)` to a stack
of derivatives `(N, d)`.

In [ ]:
f = damped_pendulum(g_over_l=1.0, damping=1.0)   # batched field (N,2) -> (N,2)
f(np.array([[0.3, -0.1], [0.0, 0.5]]))           # quick sanity evaluation

## 2. Why `L_method="evt"` here

The certificate needs a one-sided Lipschitz constant `L = sup mu(df/dx)` over the
reachable set. The default estimator (`"corners"`) evaluates the matrix measure
only at the box corners — which is **exact only when the Jacobian is affine in the
state**. The pendulum's Jacobian is

$$\frac{\partial f}{\partial x} = \begin{bmatrix} 0 & 1 \\ -\cos\theta & -1 \end{bmatrix},$$

whose `-cos(theta)` term is nonlinear. On a small box the corner value happens to
be right, but push the box past `theta = pi` and corners *misses* the interior
peak and under-estimates `L` (an unsound certificate). So we use the EVT
(reverse-Weibull) estimator, which samples the interior and returns a
**high-probability upper bound** (confidence `rho`).

In [ ]:
report = verify_stability(f, R=0.8, d=2, tau=6.0,
                          L_method="evt", rho=0.95,
                          delta=0.2, max_refine=6, record_trace=True)
print(report.summary())

## 3. Reading the report

- `report.certified` — the headline boolean (positive rate **and** the
  discretization-robustness check passed).
- `report.alpha` — the guaranteed rate (a sound *lower* bound).
- `report.alpha_upper` — the data-driven *ceiling*; the gap to `alpha` is how much
  more refinement could gain.
- `report.lipschitz.evt` — the EVT diagnostics: the fitted location parameter
  `gamma`, the KS goodness-of-fit p-value, and whether the reverse-Weibull model
  was `validated`.

In [ ]:
print("certified        :", report.certified)
print("alpha (lower bnd):", round(report.alpha, 4))
print("alpha_upper      :", round(report.alpha_upper, 4))
print("L (one-sided)    :", round(report.L, 4))
print("eq-(38) check    :", report.discretization_ok)
print("EVT:", report.lipschitz.evt.summary())

## 4. Visualize

Two reusable helpers from `pyddrv.viz`:

- `plot_stability_2d` — the verified box `Q_R` over a phase portrait of the field.
- `plot_anytime` — the certified rate as a function of wall time (a sound lower
  bound at *every* point; more compute only tightens it toward the ceiling).

In [ ]:
from pyddrv.viz import plot_stability_2d, plot_anytime

ax = plot_stability_2d(report, f=f)
ax.set_xlabel(r"$\theta$"); ax.set_ylabel(r"$\omega$")
ax.set_title("Damped pendulum: certified exponential decay on $Q_R$")
plt.show()

In [ ]:
ax = plot_anytime(report)
ax.set_title("Anytime certified rate")
plt.show()

### Seeing the covering grid

`show_grid=True` overlays the actual cubes the certifier tiled `Q_R` with, three
ways:

- `"outline"` — the tiling drawn over the phase portrait: note the **layered
  grid** (coarse cubes far from the origin, geometrically finer near it) plus the
  extra **adaptive refinement** where the rate is limited;
- `"width"` — cubes filled by their actual **width** (log scale), so the grid
  sizing is read directly;
- `"alpha"` — cubes filled by their own certified rate, showing *which* cubes
  (the darker ones) bind the global guarantee `min alpha`.

In [ ]:
plot_stability_2d(report, f=f, show_grid=True, grid_color_by="outline")
plt.title("Certified covering grid over the flow"); plt.show()

plot_stability_2d(report, show_grid=True, grid_color_by="width")
plt.title("Cube widths (log scale)"); plt.show()

plot_stability_2d(report, show_grid=True, grid_color_by="alpha")
plt.title("Per-cube certified rate"); plt.show()

## Takeaways

- `verify_stability` returned a **sound** guaranteed rate from simulated
  trajectories alone — no Lyapunov-function search.
- For this nonlinear field the EVT estimator gave a trustworthy `L`; `corners`
  would only be justified for a state-affine Jacobian (see the bilinear notebook).
- Everything is *anytime*: stopping early is valid, just conservative.